# Experiment 2 — Temporal Consistency (T05 refactor, T07 clean-up)

**Metric:** adjacent-frame cosine similarity of the upsampled features (per pixel, float64), DAVIS-2017 val (30 clips), VideoMAE backbone.
**Conditions:** `raw` (LR VideoMAE features, no upsampling), `bilinear`, `anyup2d` (per frame), `anyup3d`.

Built on the shared modules (`anyup.backbones`, `anyup.video`, `anyup.upsamplers`, `anyup.davis`, `anyup.metrics`); see `docs/plan/P1_experiments/T05_exp2_exp3_refactor.md`.

- Protocol: all frames, VideoMAE chunks of 16 with 8 frames of backbone-only overlap, output 224², AnyUp3D `q_chunk_size=4096`. Temporal cosine is reported for all pairs, pairs within a chunk, and pairs crossing a chunk boundary.
- The T05 regression check against the old Exp 2 code (`LEGACY`, passed) was removed in T07; it is in git at commit `8d34f11`.

No feature cache: everything is recomputed each run; results go to Drive (`<OUTPUT_ROOT>/<RUN_NAME>/exp2_per_item.csv` + `exp2_summary.json`). Seeding is deterministic (`anyup.utils.seed`). T4 is enough. **Runtime → Restart session** before running if `anyup` was imported before.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T10 visual checks
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
N_CLIPS      = None    # None = all 30 val clips; int = first N (smoke test). In RUN_NAME, so a smoke run never overwrites a full one
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # step6000 vs 6500 still open (plan.md)
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T05/exp2"
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0

CONDITIONS   = ["raw", "bilinear", "anyup2d", "anyup3d"]
BASELINES    = ["raw", "bilinear", "anyup2d"]   # paired_stats: anyup3d vs each of these (if run)

GUIDE_SIZE   = 224     # ↓ guide = output size of every upsampler → memory/time of all of them.
                       #   Dependent: Q_CHUNK (reduce it when this grows), GPU memory of one chunk output (8·768·GUIDE_SIZE²·4 B)
CHUNK_SIZE   = 16      # fixed by VideoMAE (NUM_FRAMES=16 position embeddings); 16 frames → 8 feature frames
OVERLAP      = 8       # ↓ backbone-only context frames shared with the previous chunk; cost × 16/(16−OVERLAP).
                       #   Dependent: CHUNK_SIZE (must be < it), VideoMAE temporal_stride=2 (must be a multiple)
Q_CHUNK      = 4096    # ↓ AnyUp3D query chunk → peak GPU memory (T03: 4096 vs None max_abs 3.8e-6 at 112²).
                       #   Dependent: GUIDE_SIZE, CHUNK_SIZE

RUN_VISUALS  = False   # optional figures + videos (Cell 6, T10). Standalone: run Cells 1–3, then 6 (reads the saved CSV)
VIZ_RUN_DIR  = None    # None = OUT_DIR; else a saved run folder, e.g. f"{OUTPUT_ROOT}/new_g224_c16_o8_q4096" (T05 run)
VIZ_ITEMS    = None    # None = auto (largest AnyUp3D − AnyUp2D gains + the smallest); or a list of clip names
N_VIZ_CLIPS  = 3       # ↓ clips re-run through backbone + 3 upsamplers (≈ main-loop time per clip). GPU memory as the main loop
VIZ_N_FRAMES = 6       # ↓ columns of the static figure (consecutive feature frames from the middle of the clip)
PCA_SAMPLE   = 50_000  # ↓ tokens per condition used to FIT PCA; fit speed + CPU RAM only (50k·768·8 B ≈ 0.3 GB each)
VIDEO_FPS    = 6       # video speed; feature frames are every 2nd input frame (VideoMAE stride 2) → 12 input fps

print(f"CONDITIONS={CONDITIONS} N_CLIPS={N_CLIPS}")

In [ ]:
# ── Cell 3 · Imports, data, models ─────────────────────────────────────────
import random, time, math
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from anyup import davis, metrics
from anyup.backbones import BackboneLoader
from anyup.video import TensorSource, iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed   # shared: seeds + deterministic cuDNN / algorithms (T07)

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0))

ROOT  = davis.download_davis(DAVIS_DEST)
CLIPS = davis.list_clips(ROOT, "val")
if N_CLIPS is not None:
    CLIPS = CLIPS[:N_CLIPS]
print(f"{len(CLIPS)} clips: {CLIPS}")
RUN_NAME = f"g{GUIDE_SIZE}_c{CHUNK_SIZE}_o{OVERLAP}_q{Q_CHUNK}_n{len(CLIPS)}"   # clip count: a smoke run can't overwrite a full run
OUT_DIR  = f"{OUTPUT_ROOT}/{RUN_NAME}"
print("RUN_NAME:", RUN_NAME)

bb  = BackboneLoader.load("videomae", device=DEVICE, guide_size=GUIDE_SIZE)
UPS = {}
if "bilinear" in CONDITIONS:
    UPS["bilinear"] = UpsamplerLoader.load("bilinear", device=DEVICE)
if "anyup2d" in CONDITIONS:
    UPS["anyup2d"] = UpsamplerLoader.load("anyup2d", device=DEVICE)
if "anyup3d" in CONDITIONS:
    UPS["anyup3d"] = UpsamplerLoader.load("anyup3d", ckpt_path=CKPT_ANYUP3D, device=DEVICE, q_chunk_size=Q_CHUNK)
print("upsamplers:", list(UPS))

In [ ]:
# ── Cell 4 · Main loop ──────────────────────────────────────────────────────
def clip_source(clip):
    """All frames of the clip."""
    return davis.open_clip(ROOT, clip)                                    # frames decoded on demand, [0, 1]

rows, frame_checks = [], []
t_start = time.perf_counter()
for ci, clip in enumerate(CLIPS):
    src = clip_source(clip)
    n_frames = len(src)
    # backbone ONCE per clip; chunks stay on the GPU (≈ 10 MB each at 224²)
    chunks = list(iter_backbone_chunks(bb, src, chunk_size=CHUNK_SIZE, overlap=OVERLAP))
    fidx = [f for ch in chunks for f in ch.meta["frame_index"]]          # source frame per feature frame, in order
    frame_checks.append(fidx == list(range(0, n_frames, bb.temporal_stride)))
    n_tokens = len(fidx)

    for cond in CONDITIONS:
        t0 = time.perf_counter()
        stream = metrics.TemporalCosineStream()
        for ch in chunks:
            # (t, 768, 14, 14) raw | (t, 768, GUIDE_SIZE, GUIDE_SIZE) upsampled, on the GPU
            feats = ch.features if cond == "raw" else UPS[cond](ch).features
            stream.update(feats, new_chunk=ch.meta["chunk_id"] > 0)
            del feats
        r = stream.result()
        rows.append(dict(clip=clip, condition=cond, all=r["all"], within=r["within"], cross=r["cross"],
                         n_pairs=r["n_pairs"], n_cross=r["n_cross"], n_frames=n_frames, n_tokens=n_tokens,
                         n_chunks=len(chunks), time_s=round(time.perf_counter() - t0, 2)))
        fmt = lambda v: "—" if v is None else f"{v:.4f}"
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<20} {cond:<9} all={r['all']:.4f} within={fmt(r['within'])} "
              f"cross={fmt(r['cross'])} frames={n_frames} tokens={n_tokens} chunks={len(chunks)} "
              f"{time.perf_counter() - t0:.1f}s")
    del chunks
    torch.cuda.empty_cache()
print(f"done in {time.perf_counter() - t_start:.0f}s | frame_index == range(0, N, 2) for every clip: {all(frame_checks)}")

In [ ]:
# ── Cell 5 · Summary, paired statistics, save ──────────────────────────────
summary = {"per_condition": {}, "paired_anyup3d_vs": {}, "frame_index_ok": all(frame_checks)}
print(f"{'condition':<10} {'all':>16} {'within':>16} {'cross':>16}")
for cond in CONDITIONS:
    rs = [r for r in rows if r["condition"] == cond]
    s = {}
    for key in ("all", "within", "cross"):
        v = [r[key] for r in rs if r[key] is not None]
        s[key] = {"mean": float(np.mean(v)) if v else None, "std": float(np.std(v)) if v else None, "n": len(v)}
    summary["per_condition"][cond] = s
    fmt = lambda d: "—" if d["mean"] is None else f"{d['mean']:.4f}±{d['std']:.4f}"
    print(f"{cond:<10} {fmt(s['all']):>16} {fmt(s['within']):>16} {fmt(s['cross']):>16}")

if "anyup3d" in CONDITIONS:
    ours = {r["clip"]: r["all"] for r in rows if r["condition"] == "anyup3d"}
    for base in BASELINES:
        if base not in CONDITIONS:
            continue
        b = {r["clip"]: r["all"] for r in rows if r["condition"] == base}
        st = metrics.paired_stats([ours[c] for c in CLIPS], [b[c] for c in CLIPS])
        summary["paired_anyup3d_vs"][base] = st
        fp = lambda p: "—" if p is None else f"{p:.2e}"
        print(f"anyup3d vs {base:<9}: Δ={st['mean_delta']:+.4f} improved {st['n_improved']}/{st['n']} "
              f"sign p2={fp(st['sign_p_two_sided'])} wilcoxon p2={fp(st['wilcoxon_p_two_sided'])}")

config = dict(N_CLIPS=N_CLIPS, clips=CLIPS, CKPT_ANYUP3D=CKPT_ANYUP3D, SEED=SEED,
              CONDITIONS=CONDITIONS, GUIDE_SIZE=GUIDE_SIZE, CHUNK_SIZE=CHUNK_SIZE, OVERLAP=OVERLAP,
              Q_CHUNK=Q_CHUNK, backbone="videomae", RUN_NAME=RUN_NAME,
              anyup3d_meta={k: v for k, v in UPS["anyup3d"]._extra_meta().items()} if "anyup3d" in UPS else None,
              anyup2d_meta=UPS["anyup2d"]._extra_meta() if "anyup2d" in UPS else None,
              repo_branch=REPO_BRANCH)
csv_path, json_path = metrics.save_results(OUT_DIR, rows, summary, config, name="exp2")
print("saved:", csv_path, json_path)

In [ ]:
# ── Cell 6 · Optional: temporal-stability figures + videos (T10) ───────────
# Standalone: run Cells 1–3, then this cell (clip choice from Cell 4's rows, or the saved CSV).
# Per selected clip: backbone + upsamplers once more, chunk by chunk; only PCA colours (uint8) and
# flicker maps (1 − per-pixel cos between consecutive feature frames) are kept, never the full features.
if RUN_VISUALS:
    import csv, json
    import cv2
    from anyup import viz
    from anyup.backbones import IMAGENET_MEAN, IMAGENET_STD
    subprocess.run(["pip", "install", "imageio-ffmpeg", "-q"], check=True)   # H.264 writer for viz.write_mp4

    run_dir = Path(VIZ_RUN_DIR or OUT_DIR)
    vis_dir = run_dir / "visuals"
    vis_dir.mkdir(parents=True, exist_ok=True)
    conds = [c for c in ("bilinear", "anyup2d", "anyup3d") if c in UPS]
    base  = "anyup2d" if "anyup2d" in conds else conds[0]                  # strongest baseline = the gain reference

    # ── per-clip temporal cosine (`all`) → clip selection ──
    if VIZ_RUN_DIR is None and globals().get("rows"):
        src_rows, source = rows, "in-memory rows (Cell 4)"
    else:
        with open(run_dir / "exp2_per_item.csv") as f:
            src_rows = list(csv.DictReader(f))
        source = str(run_dir / "exp2_per_item.csv")
    ALL = {(r["condition"], r["clip"]): float(r["all"]) for r in src_rows}
    avail = [c for c in dict.fromkeys(r["clip"] for r in src_rows) if all((k, c) in ALL for k in conds)]
    gain = lambda c: ALL["anyup3d", c] - ALL[base, c]
    if VIZ_ITEMS:
        sel, why = list(VIZ_ITEMS), {c: "manual (VIZ_ITEMS)" for c in VIZ_ITEMS}
    else:
        ranked = sorted(avail, key=gain, reverse=True)
        sel = ranked[:max(N_VIZ_CLIPS - 1, 0)] + [ranked[-1]]
        why = {c: f"gain rank {i + 1}/{len(ranked)}" for i, c in enumerate(ranked)}
        why[ranked[-1]] = f"SMALLEST gain (weakest case), rank {len(ranked)}/{len(ranked)}"
    print(f"temporal cosine from {source}")
    for c in sel:
        print(f"  {c:<20} " + " ".join(f"{k}={ALL[k, c]:.4f}" for k in conds) + f"  Δ vs {base} {gain(c):+.4f}  [{why[c]}]")

    checks = {}
    for clip in sel:
        t0 = time.perf_counter()
        chunks = list(iter_backbone_chunks(bb, clip_source(clip), chunk_size=CHUNK_SIZE, overlap=OVERLAP))
        fidx   = [f for ch in chunks for f in ch.meta["frame_index"]]             # input frame per feature frame
        starts = np.cumsum([0] + [ch.features.shape[0] for ch in chunks])[1:-1]   # feature index where a chunk starts
        guide  = np.concatenate([viz.denorm_guide(ch.guide, IMAGENET_MEAN, IMAGENET_STD) for ch in chunks])  # (N, G, G, 3)
        N = len(fidx)

        # PCA basis: joint over conditions, fitted on the first chunk; colour scale fixed per condition
        first = {k: UPS[k](chunks[0]).features for k in conds}                    # each (t, 768, G, G) GPU
        pca = viz.FeaturePCA(seed=SEED).fit({k: viz.FeaturePCA.sample_tokens(f, PCA_SAMPLE, SEED) for k, f in first.items()})
        del first

        rgb, flick, pair_mean = {}, {}, {}
        for k in conds:
            r_, f_, m_, prev = [], [], [], None
            for ch in chunks:
                f = UPS[k](ch).features                                           # (t, 768, G, G) GPU
                r_.append(pca.rgb(f, k))                                          # (t, G, G, 3) uint8
                g = f if prev is None else torch.cat([prev[None], f])             # + previous chunk's last frame
                if g.shape[0] >= 2:
                    fm = viz.flicker_map(g)                                       # (pairs, G, G) float64
                    m_.extend(fm.mean(axis=(1, 2)).tolist())
                    f_.append(fm.astype(np.float32))
                prev = f[-1].clone()
                del f, g
            rgb[k], flick[k], pair_mean[k] = np.concatenate(r_), np.concatenate(f_), np.array(m_)   # (N, ..), (N-1, ..), (N-1,)
            torch.cuda.empty_cache()

        # check: mean flicker = 1 − the experiment's temporal cosine of this clip (same pairs, same math)
        for k in conds:
            d = abs((1 - pair_mean[k].mean()) - ALL[k, clip])
            checks[f"{clip}/{k}"] = d
            print(f"  check {clip}/{k}: |1 − mean flicker − saved cos| = {d:.2e} {'PASS' if d < 1e-6 else 'WARN (> 1e-6)'}")

        vmax = float(np.percentile(np.concatenate([flick[k][::4].ravel() for k in conds]), 99))   # shared colour scale

        # ── static figure: input row, then PCA + flicker rows per condition, VIZ_N_FRAMES middle frames ──
        K  = min(VIZ_N_FRAMES, N)
        s0 = max(1, (N - K) // 2)                                                 # ≥ 1 so every column has a flicker pair
        cols = list(range(s0, min(s0 + K, N)))
        labels = ["input"] + [lab for k in conds for lab in (f"{k}\nPCA", f"{k}\nchange\ncos {ALL[k, clip]:.3f}")]
        fig, axes = plt.subplots(len(labels), len(cols), figsize=(2.3 * len(cols) + 1.6, 2.3 * len(labels)), squeeze=False)
        for j, t in enumerate(cols):
            boundary = " | new chunk" if t in starts else ""
            axes[0, j].imshow(guide[t]); axes[0, j].set_title(f"frame {fidx[t]}{boundary}", fontsize=9)
            for r, k in enumerate(conds):
                axes[1 + 2 * r, j].imshow(rgb[k][t])
                im = axes[2 + 2 * r, j].imshow(flick[k][t - 1], cmap="inferno", vmin=0, vmax=vmax)
        for r, lab in enumerate(labels):
            for ax in axes[r]:
                ax.set_xticks([]); ax.set_yticks([])
            axes[r, 0].set_ylabel(lab, fontsize=10, fontweight="bold" if "anyup3d" in lab else "normal")
        fig.subplots_adjust(left=0.08, right=0.9, top=0.94, bottom=0.01, wspace=0.05, hspace=0.12)
        cax = fig.add_axes([0.92, 0.1, 0.015, 0.6])                              # own axis: panels keep equal size
        fig.colorbar(im, cax=cax, label="1 − cos(previous frame, this frame)   (dark = stable)")
        fig.suptitle(f"Exp 2 · {clip}: VideoMAE features upsampled to {GUIDE_SIZE}² — colours = PCA of the features, "
                     f"'change' = how much each pixel's feature changes from the previous frame", fontsize=11, y=0.985)
        fig_path = vis_dir / f"exp2_{clip}.png"
        plt.savefig(fig_path, dpi=130, bbox_inches="tight"); plt.show()

        # ── curve: temporal cosine of every consecutive pair, chunk starts dashed ──
        fig, ax = plt.subplots(figsize=(10, 3.2))
        for k in conds:
            ax.plot(fidx[1:], 1 - pair_mean[k], label=f"{k} (mean {ALL[k, clip]:.3f})", lw=2 if k == "anyup3d" else 1.2)
        for s in starts:
            ax.axvline(fidx[s], color="gray", ls="--", lw=0.8)
        ax.set_xlabel("input frame (dashed = AnyUp3D chunk start)"); ax.set_ylabel("cos(frame t−1, frame t)")
        ax.set_title(f"Exp 2 · {clip}: frame-to-frame feature similarity (higher = more stable)")
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
        curve_path = vis_dir / f"exp2_{clip}_curve.png"
        plt.savefig(curve_path, dpi=130, bbox_inches="tight"); plt.show()

        # ── video: top = input + PCA per condition, bottom = legend + flicker per condition ──
        G = guide.shape[1]
        legend = np.zeros((G, G, 3), np.uint8)
        legend[G // 2 - 10:G // 2 + 10, 12:G - 12] = viz.colorize(np.tile(np.linspace(0, vmax, G - 24), (20, 1)), 0, vmax)
        cv2.putText(legend, "change from previous frame", (8, G // 2 - 24), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(legend, "stable", (10, G // 2 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(legend, "changing", (G - 82, G // 2 + 30), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        flick_rgb = {k: viz.colorize(flick[k], 0, vmax) for k in conds}           # (N-1, G, G, 3)
        frames = []
        for t in range(N):
            top = [viz.label_strip(guide[t], f"input frame {fidx[t]}")] + \
                  [viz.label_strip(rgb[k][t], f"{k} features") for k in conds]
            bot = [viz.label_strip(legend, "")] + \
                  [viz.label_strip(flick_rgb[k][t - 1] if t > 0 else np.zeros_like(legend), f"{k} cos {ALL[k, clip]:.3f}")
                   for k in conds]
            frames.append(viz.label_strip(viz.grid([top, bot]), f"{clip}: top = feature colours (PCA), bottom = "
                                          f"change from previous frame (dark = stable)", height=26))
        mp4 = viz.write_mp4(frames, vis_dir / f"exp2_{clip}.mp4", fps=VIDEO_FPS)
        gif = viz.write_gif(frames, vis_dir / f"exp2_{clip}.gif", fps=VIDEO_FPS)
        print(f"saved {fig_path.name}, {curve_path.name}, {mp4.name}, {gif.name}  ({N} feature frames, "
              f"{time.perf_counter() - t0:.0f}s)")
        del chunks, rgb, flick, flick_rgb, frames
        torch.cuda.empty_cache()

    with open(vis_dir / "visuals_selection.json", "w") as f:
        json.dump(dict(source=source, base=base, flicker_vs_saved_cos_absdiff=checks,
                       clips=[dict(clip=c, why=why[c], **{k: ALL[k, c] for k in conds}) for c in sel]), f, indent=2)
    print("saved figures + videos + visuals_selection.json in", vis_dir)
else:
    print("RUN_VISUALS=False — skipped")